# Textbook canonical — Durbin's dishonest casino

From *Biological Sequence Analysis* (Durbin, Eddy, Krogh, Mitchison, Cambridge 1998), Chapter 3.

A casino occasionally swaps a fair die for a loaded one that prefers face 6. The dealer's swaps follow a 2-state Markov chain. Given a long sequence of rolls, can Viterbi recover **when** the loaded die was in play ?

This is the standard sanity-check for any HMM Viterbi implementation : the problem is genuinely hard (loaded looks like fair most of the time), but a correct decoder should still significantly beat the always-fair baseline.

## 1. The canonical model

Parameters from Durbin et al., Chap. 3 :

- Hidden states : **fair** (0), **loaded** (1)
- Observations : die faces 1..6 (encoded as 0..5)
- `P(stay in fair)` = 0.95
- `P(stay in loaded)` = 0.90
- Fair die : uniform 1/6 per face
- Loaded die : `[0.1, 0.1, 0.1, 0.1, 0.1, 0.5]` (favors face 6)

In [ ]:
import numpy as np
from hmm_core.fit.multinomial import ConstrainedMultinomialHMM

model = ConstrainedMultinomialHMM(
    n_components=2, n_features=6, random_state=0, transmat_mask=None,
)
model.startprob_ = np.array([0.5, 0.5])
model.transmat_ = np.array([
    [0.95, 0.05],
    [0.10, 0.90],
])
model.emissionprob_ = np.array([
    [1 / 6, 1 / 6, 1 / 6, 1 / 6, 1 / 6, 1 / 6],   # fair : uniform
    [0.10,  0.10,  0.10,  0.10,  0.10,  0.50 ],   # loaded : favors face 6
])

## 2. Sample a 1000-roll sequence from the model

By the stationary distribution of the transition chain, the *true* fraction of loaded rolls is roughly 1/3 (since `π_fair · 0.05 = π_loaded · 0.10` ⇒ `π_loaded = 1/3`).

In [ ]:
T = 1000
X, Z_true = model.sample(T, random_state=42)
loaded_fraction = (Z_true == 1).mean()
print(f"T = {T}")
print(f"True loaded fraction : {loaded_fraction:.2%}  (theory : ~33%)")
print(f"Frequency of face 6 overall : {(X[:, 0] == 5).mean():.2%}")

## 3. Viterbi decode and accuracy

Run Viterbi to recover the most likely state sequence. We take the **max over both label permutations** (the HMM doesn't know which state we call 'fair') to compute accuracy.

The always-fair baseline gets ~2/3 = 67%. Durbin Figure 3.5 shows Viterbi catching most long loaded runs and missing short ones, typically scoring 70-80%. Our threshold is 70% — significantly above baseline.

In [ ]:
Z_pred = model.predict(X)
acc_direct = float(np.mean(Z_pred == Z_true))
acc_swap = float(np.mean(Z_pred == (1 - Z_true)))
accuracy = max(acc_direct, acc_swap)

print(f"Direct alignment  : {acc_direct:.2%}")
print(f"Swapped alignment : {acc_swap:.2%}")
print(f"Best Viterbi accuracy : {accuracy:.2%}")
print(f"Always-fair baseline  : {(Z_true == 0).mean():.2%}")
assert accuracy >= 0.70, "Viterbi accuracy below 70% baseline-plus-margin threshold"
print("\nOK — exceeds the Durbin 70% threshold.")

## 4. Sanity check : decoded 'loaded' state is skewed toward face 6

If our Viterbi is doing the right thing, then conditional on the predicted-loaded label, face 6 should appear way more often than 1/6 — close to the loaded die's design value of 50%.

In [ ]:
# Identify the predicted-loaded label by max face-6 frequency
face6_per_state = [
    (X[Z_pred == k, 0] == 5).mean() if (Z_pred == k).any() else 0.0
    for k in range(2)
]
loaded_label = int(np.argmax(face6_per_state))

p6_in_loaded = face6_per_state[loaded_label]
p6_in_fair = face6_per_state[1 - loaded_label]
print(f"P(face 6 | decoded loaded) : {p6_in_loaded:.2%}  (target ~50%)")
print(f"P(face 6 | decoded fair)   : {p6_in_fair:.2%}   (target ~17%)")

## Why this matters

The dishonest casino is one of the most-cited HMM sanity checks in the literature. Reproducing Durbin's results in our own code, against a fresh sampled sequence, is how we tell reviewers : **the Viterbi we ship is not a custom variant, it's the textbook algorithm, and it behaves as published**.

Same family of validation : the AIMA umbrella world in `07_textbook_aima_umbrella.ipynb`, and the Eisner ice cream HMM (Jurafsky teaching) in `validation/test_v3_textbook_canonical.py::test_v3_3_eisner_ice_cream_*`.